# 2.3 Calculate degassing path using Sulfur_X

<div style="background-color:#eef8fa; border-left:4px solid #24bdff; padding:10px; border-radius:4px;">
<b> 🧮 &nbsp; Sulfur_X </b> is a model of sulfur degassing during magma ascent.

More information on Sulfur_X can be found at https://github.com/sdecho/Sulfur_X

</div>

This example has been modified from the ``Sulfur_X.ipynb`` notebook in the shared workflows from VICTOR.

## 2.3.1 Import packages and note versions

In [ ]:
# Packages that we will use in our code always get imported before we need them.
# This is canonically done at the top of a script.
# ⚠️ Note that this can take a few seconds if it's the first time you're importing these libraries.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
os.makedirs("output", exist_ok=True) # creates the output folder

 We have to define the directory where Sulfur_X is hosted - in this case its within the folder that hosts this notebook. This notebook was created using Sulfur_X: 1.2.

In [ ]:
SulfurX_directory = 'Sulfur_X-main/'

## 2.3.2 Import data

We'll use the same melt inclusion dataset from notebook <a href="1_1_MI_Temperature_Thermobar.ipynb">1.1 Calculate temperature from MI data using Thermobar</a>, where we've already calculated temperatures.

In [ ]:
# import melt inclusion dataset
MI = pd.read_csv("output/wieser2021_w_temperatures.csv")

# if you haven't run Exercise 1, you can grab the "answer key" file from here - remove the # at the start of the line below and then run the cell
#results_pvsat_vf = pd.read_csv("answers/wieser2021_w_temperatures.csv")

## 2.3.3 Explore options

As with the other tools, Sulfur_X has different model options available, which can be chosen using the following parameters. Here we'll use the CO<sub>2</sub>-H<sub>2</sub>O solubility of Iacono-Marziano et al. (2012) to match the VESIcal calculations and the S speciation model of O'Neil & Mavrogenes (2022) to match the VolFe calculations. Crystallisation and sulfide precipitation are turned off and the calculations are closed-system degassing. We do not perform Monte Carlo simulations.

In [ ]:
# Which COH degassing model to use? 1 for VolatileCalc; 0 for Iacono_Marziano model
COH_model = 0

# Which S speciation model to use? 0 for Nash model; 1 for O'Neill and Mavrogenes (2022) model; 100 for Muth model
S_Fe_choice = 1

# The options below are NOT being changed for this calculation

# Changing fO2 or not? If 1, fO2 changes by S degassing and S-Fe electron exchange;
# if 0, fO2 if buffered at the delta_FMQ
fo2_tracker = 1
# Enable crystallization (yes=1,no=0)
crystallize = 0
# sulifde saturation: if 0, no sulfide precipitation; if 1, sulfide precipitation enabled
sulfide_pre = 0
# Use monte carlo simulation for error estimate? (1/0)
# If 1, input number of runs, otherwise ignore
monte_carlo = 0
mc_runs = 0
# Decimal value between 0 and 1, percentage of melt; initial value to start the calculation. 
melt_fraction = 1
# degassing style: if 0, fully closed degassing; if x, degassing become open when pressure is lower than xMPa
open_degassing = 0

The parameters in the cell below effect how the calculation is performed - more information on these can be found on the Sulfur_X GitHub.

In [ ]:
# The total steps of pressure from initial P to 1 bar.
l = 1000

# Total number of runs along degassing
m = 1000

# Pressure threshold (in bar) below which the Kd_combined can increase with an arbiturary number defined by INC
BAR = 0

# Arbiturary increase of Kd_combined when pressure is lower than BAR (in bar) when BAR > 0; if BAR = 0, INC is not relevant
INC = 20

# Tolerance (sigma) of log_10(fO2)
sigma = .01

These final options are about where the data are saved to and where MI data are taken from for plotting.

In [ ]:
# prepare MI file
MI_sfx = MI.copy()
MI_sfx = MI_sfx.rename(columns = {"CO2_ppm":"mi_CO2","S":"mi_S","H2O":"mi_H2O","SaturationP_bars_VESIcal":"pressure"})
MI_sfx.to_csv(SulfurX_directory+'Kilauea.csv')

# Melt inclusion data file (.csv) Sulfur_X can read in MI file (template shown by Fuego.csv) for comparison. Here we will use the Kilauea file that we have been working on instead.
mi_name = SulfurX_directory+"Kilauea.csv"

# Output path
output_path = SulfurX_directory+'results_folder'

#Save plot images? (True/False)
save_img = False

## 2.3.4 Initial melt composition

We choose the MI with the highest CO<sub>2</sub> content as the starting point for the degassing calcuations

In [ ]:
# row number with highest CO2 content
row = MI['CO2'].idxmax()

print(f"The sample with the highest CO2 content is {row} with {MI['CO2'][row]} wt%")

We also specify the initial <i>f</i>O<sub>2</sub> (ΔFMQ = 0.2 as before)

In [ ]:
DFMQ = 0.2

And combine this all into a single dictionary to use in the calculations in the following sections

In [ ]:
ini_comp = MI.loc[row].to_dict() # convert to dictionary
ini_comp = {k.split(" ")[0]: v for k, v in ini_comp.items()}
ini_comp.update({
    'DFMQ': DFMQ
})

# print initial composition
ini_comp

And set the composition for Sulfur_X. We won't use it today, but the sulfide composition can be defined as well as the sulfur isotope ratio (these can also be used in VolFe). If crystallisation is required, this should also be defined - we are not using this option today.

In [ ]:
# Temperature in C
temperature = round(ini_comp['T_C'])

# fO2 relative to FMQ buffer
delta_FMQ = ini_comp['DFMQ']

# initial H2O in wt.%
H2O_initial = ini_comp['H2O']

# initial CO2 in ppm
CO2_initial = round(ini_comp['CO2_ppm'])

# initial sulfur in ppm
S_initial = ini_comp['S']

# melt composition
sio2 = float(ini_comp['SiO2'])
al2o3 = float(ini_comp['Al2O3'])
feot = float(ini_comp['FeO']+ini_comp['Fe2O3']*0.8998)
mgo = float(ini_comp['MgO'])
cao = float(ini_comp['CaO'])
na2o = float(ini_comp['Na2O'])
k2o = float(ini_comp['K2O'])
p2o5 = float(ini_comp['P2O5'])
mno = float(ini_comp['MnO'])
tio2 = float(ini_comp['TiO2'])

# Options below are NOT being used in the example today

# Sulfide composition in wt%, only relevant if SCSS is of interest. 
sulfide = {"Fe": 65.43,
            "Ni": 0,
            "Cu": 0,
            "O": 0,
            "S": 36.47
            }
# initial sulfur isotope ratio
d34s_m_initial = 1
# if crystallization is enabled, H2O-melt fraction relation is specified using H2O-K2O relation (K2O = a * H2O +b),
# assuming K2O is perfectly incompatible. The given a and b are based on H2O-K2O relation for Fuego magma from
# Lloyd et al. (2013). Both H2O and K2O are in wt.%. If crystallization is disabled, or running on magmas similar to Fuego,
# leave them unchanged.
slope_h2o = -0.713
constant_h2o = 3.689

## 2.3.5 Run calculation

We input all these values into the .py files for Sulfur_X.

In [ ]:
f = open(SulfurX_directory+"main_Fuego.py","r+")
main = f.readlines()
main[32] = f"""temperature = {temperature}\n"""
main[35] = f"""delta_FMQ = {delta_FMQ}\n"""
main[37] = f"""H2O_initial = {H2O_initial}\n"""
main[39] = f"""CO2_initial = {CO2_initial}\n"""
main[41] = f"""S_initial = {S_initial}\n"""
main[43] = f"""d34s_m_initial = {d34s_m_initial}\n"""
main[45] = f"""choice = {crystallize}\n"""
main[47] = f"""COH_model = {COH_model}\n"""
main[50] = f"""fo2_tracker = {fo2_tracker}\n"""
main[54] = f"""monte_carlo = {monte_carlo}\n"""
main[57] = f"""    m_run = {mc_runs}\n"""
main[63] = f"""l = {l}\n"""
main[65] = f"""m = {m}\n"""
main[68] = f"""sulfide = {{"Fe": {sulfide["Fe"]},\n"""
main[69] = f"""         "Ni": {sulfide["Ni"]},\n"""
main[70] = f"""         "Cu": {sulfide["Cu"]},\n"""
main[71] = f"""         "O": {sulfide["O"]},\n"""
main[72] = f"""         "S": {sulfide["S"]},\n"""
main[75] = f"""open_degassing = {open_degassing}\n"""
main[77] = f"""sulfide_pre = {sulfide_pre}\n"""
main[85] = f"""S_Fe_choice = {S_Fe_choice}\n"""
main[88] = f"""sigma = {sigma}\n"""
main[94] = f"""slope_h2o = {slope_h2o}\n"""
main[95] = f"""constant_h2o = {constant_h2o}\n"""
main[102] = f"""mi_name = '{mi_name}'\n"""
main[107] = f"""folder = Path("{output_path}")\n"""
f.seek(0)
f.writelines(main)
f.truncate()
f.close()

g = open(SulfurX_directory+"degassingrun.py","r+")
degas = g.readlines()
degas[13] = f"""INC = {INC}\n"""
degas[14] = f"""BAR = {BAR}\n"""
degas[35] = f"""        self.sulfide = {{"Fe": {sulfide["Fe"]},\n"""
degas[36] = f"""                        "Ni": {sulfide["Ni"]},\n"""
degas[37] = f"""                        "Cu": {sulfide["Cu"]},\n"""
degas[38] = f"""                        "O": {sulfide["O"]},\n"""
degas[39] = f"""                        "S": {sulfide["S"]},\n"""
g.seek(0)
g.writelines(degas)
g.truncate()
g.close()

g = open(SulfurX_directory+"melt_composition.py","r+")
melt_comp = g.readlines()
melt_comp[76] = f"""            sio2 = {sio2} / melt_fraction\n"""
melt_comp[77] = f"""            al2o3 = {al2o3} / melt_fraction\n"""
melt_comp[78] = f"""            feot = {feot} / melt_fraction\n"""
melt_comp[79] = f"""            mgo = {mgo} / melt_fraction\n"""
melt_comp[80] = f"""            cao = {cao} / melt_fraction\n"""
melt_comp[81] = f"""            na2o = {na2o} / melt_fraction\n"""
melt_comp[82] = f"""            k2o = {k2o} / melt_fraction\n"""
melt_comp[83] = f"""            p2o5 = {p2o5} / melt_fraction\n"""
melt_comp[84] = f"""            mno = {mno} / melt_fraction\n"""
melt_comp[85] = f"""            tio2 = {tio2} / melt_fraction\n"""
g.seek(0)
g.writelines(melt_comp)
g.truncate()
g.close()

And run the calculation! Make sure the path is correct here - it won't change if you change it in the cells above unfortunately.

In [ ]:
%run Sulfur_X-main/main_Fuego.py

In [ ]:
# save your results
results_degas_sfx.to_csv("output/results_degas_sfx.csv")

# uncomment the line below to interrogate the resulting dataframe
#results_degas_sfx